# Exercise 4 — format_report

`format_report` turns the result dict into a human-readable string for logging and alerts. Day 96 will extend this to write to a log file and optionally send a notification. The function must include all key metrics and, if trades exist, the first and last trade.

In [ ]:
import pandas as pd, math
from dataclasses import dataclass, field

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
@dataclass
class Trade:
    date:        object
    action:      str
    price:       float
    shares:      float
    cash_after:  float
    value_after: float
@dataclass
class PaperAccount:
    initial_cash: float = 10_000.0
    cash:         float = field(init=False)
    shares:       float = field(init=False)
    trades:       list  = field(init=False)

    def __post_init__(self):
        self.cash   = self.initial_cash
        self.shares = 0.0
        self.trades = []

    def portfolio_value(self, price):
        return self.cash + self.shares * float(price)

    def buy(self, date, price, fraction=1.0):
        price = float(price)
        if self.cash <= 0 or price <= 0:
            return None
        shares = (self.cash * fraction) / price
        cost   = shares * price
        if cost > self.cash:
            shares = self.cash / price
            cost   = shares * price
        self.cash   -= cost
        self.shares += shares
        t = Trade(date=date, action="BUY", price=price, shares=shares,
                  cash_after=self.cash,
                  value_after=self.portfolio_value(price))
        self.trades.append(t)
        return t

    def sell(self, date, price):
        price = float(price)
        if self.shares <= 0:
            return None
        proceeds    = self.shares * price
        sold_shares = self.shares
        self.cash  += proceeds
        self.shares = 0.0
        t = Trade(date=date, action="SELL", price=price, shares=sold_shares,
                  cash_after=self.cash,
                  value_after=self.portfolio_value(price))
        self.trades.append(t)
        return t
def run_paper_trader(df, signals, initial_cash=10_000.0, fraction=1.0):
    account = PaperAccount(initial_cash=initial_cash)
    eq_values   = []
    prev_signal = 0
    for i in range(len(df)):
        date  = df.index[i]
        price = float(df["Close"].iloc[i])
        sig   = int(signals.iloc[i])
        if sig == 1 and prev_signal == 0:
            account.buy(date, price, fraction=fraction)
        elif sig == 0 and prev_signal == 1:
            account.sell(date, price)
        eq_values.append(account.portfolio_value(price))
        prev_signal = sig
    if account.shares > 0:
        account.sell(df.index[-1], float(df["Close"].iloc[-1]))
    equity       = pd.Series(eq_values, index=df.index)
    total_return = float(equity.iloc[-1] / initial_cash - 1.0)
    peak         = equity.cummax()
    max_dd       = float(((equity - peak) / peak).min())
    return {
        "account":      account,
        "trades":       account.trades,
        "equity":       equity,
        "initial_cash": initial_cash,
        "final_value":  float(equity.iloc[-1]),
        "total_return": total_return,
        "max_drawdown": max_dd,
        "n_trades":     len(account.trades),
        "n_buys":       sum(1 for t in account.trades if t.action == "BUY"),
        "n_sells":      sum(1 for t in account.trades if t.action == "SELL"),
    }

def format_report(result):
    """Format the run_paper_trader result as a multiline string.

    Required fields in output:
        - initial cash, final value, total return, max drawdown
        - total trades, buys, sells
        - first and last trade (date, action, price) if any trades exist

    Returns:
        str
    """
    # TODO: build a list of lines, join with "\n"
    return ""


### Checks

In [ ]:
checks = 0

# 1 — returns a non-empty string
try:
    df  = _synthetic()
    sig = pd.Series(1, index=df.index)
    r   = run_paper_trader(df, sig)
    rep = format_report(r)
    assert isinstance(rep, str) and len(rep) > 50,         f"expected a non-trivial string, got: {repr(rep)}"
    checks += 1; print("✅ 1 format_report returns a non-empty string")
except Exception as e:
    print("❌ 1:", e)

# 2 — contains key metrics as strings
try:
    df  = _synthetic()
    sig = pd.Series(1, index=df.index)
    r   = run_paper_trader(df, sig)
    rep = format_report(r)
    text = rep.lower()
    assert "initial" in text, "should mention initial cash"
    assert "return"  in text, "should mention total return"
    assert "drawdown" in text, "should mention max drawdown"
    checks += 1; print("✅ 2 report mentions initial cash, return, and drawdown")
except Exception as e:
    print("❌ 2:", e)

# 3 — total return value appears correctly formatted
try:
    df  = _synthetic()
    sig = pd.Series(1, index=df.index)
    r   = run_paper_trader(df, sig)
    rep = format_report(r)
    tr_pct = f"{r['total_return']:.2%}"
    assert tr_pct.replace("-","") in rep.replace("-",""),         f"total_return {tr_pct} not found in report"
    checks += 1; print(f"✅ 3 total return {tr_pct} appears in report")
except Exception as e:
    print("❌ 3:", e)

# 4 — flat signal: report mentions 0 trades
try:
    df  = _synthetic()
    sig = pd.Series(0, index=df.index)
    r   = run_paper_trader(df, sig)
    rep = format_report(r)
    assert "0" in rep, "report should show 0 trades for flat signal"
    checks += 1; print("✅ 4 flat signal report shows 0 trades")
except Exception as e:
    print("❌ 4:", e)

# 5 — print the full report
try:
    df  = _synthetic()
    sig = pd.Series(1, index=df.index)
    r   = run_paper_trader(df, sig, initial_cash=10_000.0)
    rep = format_report(r)
    print(rep)
    checks += 1; print("✅ 5 report printed successfully")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
